In [ ]:
import json

import numpy as np
import pystac_client
from dask.distributed import Client as DaskClient
from odc.stac import load
from planetary_computer import sign_url
from shapely.geometry import shape

In [ ]:
NUM_SAMPLES = 600
AOI_PATH = "geoBoundaries-BFA-ADM0.geojson"

In [ ]:
dask_client = DaskClient()
dask_client

In [ ]:
def create_dataset(start_date, end_date, num_samples):
    with open(AOI_PATH) as file:
        area_of_interest = json.load(file)
        area_of_interest = area_of_interest["features"][0]["geometry"]
        geom = shape(area_of_interest)
        bbox = list(geom.bounds)

    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1/"
    )
    collection = "landsat-c2-l2"

    search = catalog.search(
        collections=[collection],
        bbox=bbox,
        datetime=f"{start_date}/{end_date}",
    )
    items = search.item_collection()

    data = load(
        items,
        bands=["red", "nir08", "qa_pixel"],
        bbox=bbox,
        chunks={"x": 1024, "y": 1024, "time": 25},
        resolution=300,
        groupby="solar_day",
        patch_url=sign_url,
    )

    flat = data.stack(z=("x", "y"))
    rng = np.random.default_rng(seed=0)  # fixed seed for reproducibility
    sample_indices = rng.choice(flat.z.size, size=num_samples, replace=False)
    data = flat.isel(z=sample_indices)

    # Mask out nodata and cloud pixels
    # Bit 3 is cloud shadow, bit 4 is cloud, and bit 0 is nodata
    mask_bits = 0b00011001

    mask = (data.qa_pixel & mask_bits) != 0

    data = data.where(~mask, other=np.nan).drop_vars("qa_pixel")

    ndvi = (data.nir08 - data.red) / (data.nir08 + data.red)
    data["ndvi"] = ndvi.clip(-1, 1)
    data = data.drop_vars(["red", "nir08"])
    data = data.astype("float32", copy=False)

    eight_day = data.ndvi.resample(time="8D").max()
    eight_day = eight_day.chunk({"time": -1})
    filled = (
        eight_day.interpolate_na("time", method="linear").bfill("time").ffill("time")
    )

    filled = filled.reset_index(dims_or_levels="z")

    filled = filled.chunk({"time": 10, "z": 10000})

    return filled

In [ ]:
train_dataset = create_dataset(
    start_date="2024-01", end_date="2024-02", num_samples=NUM_SAMPLES
)

In [ ]:
train_dataset

In [ ]:
train_dataset.to_zarr("data/train_set.zarr", mode="w", consolidated=True)

In [ ]:
val_dataset = create_dataset(
    start_date="2025-01", end_date="2025-02", num_samples=NUM_SAMPLES
)

In [ ]:
val_dataset

In [ ]:
val_dataset.to_zarr("data/val_set.zarr", mode="w", consolidated=True)